#### segpriors — Kaggle worker template

Generic single-config launch template for the dashboard's Kaggle runner. **Do not edit or push
this notebook directly** — the dashboard renders a copy of it per launch, substituting the
`# DASHBOARD:LAUNCH_SPEC` cell below with a real `config_path` / `mode` / `extra_args`, then
pushes the rendered copy. One push = one config + one mode, the same shape as a local
`python train.py --config <path>` launch on mclab.

Boilerplate (GPU check, repo clone, pinned Miniconda env, dataset attach, run, package results)
is copied from this study's earlier hand-written per-worker notebooks
(`notebooks/iccit-kaggle-worker3.ipynb` / `...worker4.ipynb`) — those still work unchanged and
are not affected by this template.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import subprocess
print(subprocess.run(["python3", "--version"], capture_output=True, text=True).stdout)


## 1. Clone the repo

In [ ]:
%cd /kaggle/working
!rm -rf segpriors
!git clone https://github.com/Syfur007/segpriors.git segpriors
%cd segpriors


## 2. Reproduce the training environment (Python 3.8 + pinned deps)

Kaggle's default image is a newer Python than this repo's pinned stack (`requirements.txt` is
frozen against Python 3.8). This builds a matching Miniconda env once, so this worker runs the
same interpreter + package versions as every other device in the study — not just "close
enough".

In [ ]:
import os
if not os.path.exists("/opt/conda_iccit"):
    !wget -q https://repo.anaconda.com/miniconda/Miniconda3-py38_23.11.0-2-Linux-x86_64.sh -O /tmp/miniconda.sh
    !bash /tmp/miniconda.sh -b -p /opt/conda_iccit
!/opt/conda_iccit/bin/conda --version


In [ ]:
PY = "/opt/conda_iccit/bin/python"
PIP = "/opt/conda_iccit/bin/pip"

# --prefer-binary: a few requirements.txt packages fall back to compiling from source if pip
# can't find a prebuilt wheel for the resolved version, which can silently burn 20-40+ minutes
# of this notebook's time budget. Deliberately NOT run with -q: a quiet install has previously
# ended up silently missing a package with no error surfaced.
!{PIP} install --upgrade pip
!{PIP} install --prefer-binary torch==1.11.0+cu113 torchvision==0.12.0+cu113 --extra-index-url https://download.pytorch.org/whl/cu113
!{PIP} install --prefer-binary -r requirements.txt


### 2b. Verify every critical import actually landed

In [ ]:
CRITICAL_IMPORTS = [
    "torch", "torchvision", "numpy", "scipy", "sklearn", "skimage", "pandas",
    "loguru", "yaml", "cv2", "albumentations", "pydantic", "pyarrow",
    "statsmodels", "fvcore", "SimpleITK", "nibabel", "h5py", "tensorboard",
    "tensorboardX", "timm", "transformers", "captum", "thop",
]
failed = []
for mod in CRITICAL_IMPORTS:
    result = os.system(f'{PY} -c "import {mod}" 2>/tmp/import_err.txt')
    if result != 0:
        err = open("/tmp/import_err.txt").read().strip().splitlines()[-1:]
        failed.append((mod, err[0] if err else "unknown error"))
        print(f"  MISSING: {mod:20s} {err[0] if err else ''}")
    else:
        print(f"  ok:      {mod}")

if failed:
    raise RuntimeError(
        f"{len(failed)} package(s) failed to import after installation: {[m for m, _ in failed]}. "
        "Re-run the pip install cell above and check its full (non-quiet) output for why."
    )
print("\nAll critical imports OK.")

import torch
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())


## 3. Launch spec — filled in by the dashboard

Left as-is (unsubstituted), the next cell fails loudly instead of silently running nothing.

In [ ]:
# DASHBOARD:LAUNCH_SPEC — substituted by backend/kaggle.py's _render_launch_notebook() before
# push. Do not hand-edit these values in this template file — edit a *pushed copy* only
# if you're debugging a specific run, never this shared template.
CONFIG_PATH = "__DASHBOARD_CONFIG_PATH__"
EXTRA_ARGS = "__DASHBOARD_EXTRA_ARGS__"
# The actual scripts to run — resolved server-side from this deployment's own repo profile
# (train_script/eval_script), NOT hardcoded to train.py/eval.py here. A deployment may point
# these at a wrapper (e.g. this study's own scripts/run_iccit_sweep.py, which loops the
# pre-registered seeds and writes the manifest/ledger rows train.py's own CLI never does)
# rather than train.py/eval.py directly. Mirrors tmux_runner.build_launch_command()'s own
# settings.train_script/settings.eval_script choice, so a Kaggle-launched run and an
# mclab-launched run of the same config actually run the *same command*.
#
# No MODE: this push always runs both stages, train then eval, inside this one kernel —
# see the Run cell below.
TRAIN_SCRIPT = "__DASHBOARD_TRAIN_SCRIPT__"
EVAL_SCRIPT = "__DASHBOARD_EVAL_SCRIPT__"
# Flags always appended to the eval command only (e.g. --ensemble) — this deployment's own
# eval_default_args, mirroring how an mclab-launched eval gets the same flags.
EVAL_EXTRA_FLAGS = "__DASHBOARD_EVAL_EXTRA_FLAGS__"

assert CONFIG_PATH and not CONFIG_PATH.startswith("__DASHBOARD_"), (
    "CONFIG_PATH was never substituted -- this notebook was uploaded/run directly instead of "
    "pushed through the dashboard's Kaggle launch flow, which is what fills in this cell."
)
print("CONFIG_PATH:", CONFIG_PATH)
print("EXTRA_ARGS:", EXTRA_ARGS or "(none)")
print("TRAIN_SCRIPT:", TRAIN_SCRIPT)
print("EVAL_SCRIPT:", EVAL_SCRIPT)
print("EVAL_EXTRA_FLAGS:", EVAL_EXTRA_FLAGS or "(none)")


## 4. Attach the dataset this config needs

Generic by design: resolves `dataset.name`/`dataset.root` straight from the *composed* config
(same `utils.config.load_config()` every other device uses), then looks for a same-named
directory already attached under `/kaggle/input/` and symlinks it into place. If nothing matches,
attach the right Kaggle dataset via **+ Add Data** first — check `configs/dataset/*.yaml` for
what root layout that dataset expects (e.g. ISIC18 needs a pre-resized copy per its own fragment's
comment, not the raw challenge download).

In [ ]:
import subprocess, glob, shutil

os.environ["PYTHONPATH"] = os.getcwd()
probe = subprocess.run(
    [PY, "-c",
     "from utils.config import load_config; c = load_config(" + repr(CONFIG_PATH) + "); "
     "print(c.dataset.name); print(c.dataset.root)"],
    capture_output=True, text=True,
)
assert probe.returncode == 0, f"Could not resolve dataset info for {CONFIG_PATH}:\n{probe.stderr}"
dataset_name, dataset_root = probe.stdout.strip().splitlines()[:2]
print("dataset:", dataset_name, "-> root:", dataset_root)

if not os.path.isdir(dataset_root):
    leaf = os.path.basename(dataset_root.rstrip("/"))
    candidates = [p for p in glob.glob(f"/kaggle/input/**/{leaf}", recursive=True) if os.path.isdir(p)]
    assert candidates, (
        f"No '{leaf}' directory found under /kaggle/input/ for dataset '{dataset_name}'. "
        f"Attach the matching Kaggle dataset via '+ Add Data' first, then re-run this cell."
    )
    parent = os.path.dirname(dataset_root.rstrip("/")) or "."
    os.makedirs(parent, exist_ok=True)
    if os.path.islink(dataset_root):
        os.remove(dataset_root)
    elif os.path.exists(dataset_root):
        shutil.rmtree(dataset_root)
    os.symlink(candidates[0], dataset_root)
    print("Symlinked", dataset_root, "->", candidates[0])
else:
    print(dataset_root, "already present locally.")


## 5. Run

Train, then eval — both inside this one kernel execution (there is no separate train-only or
eval-only push; a Kaggle account only runs one kernel at a time, so "train" and "eval" as two
separate pushes could never be chained the way two local tmux sessions can). Eval only runs if
train exits 0, mirroring the scheduler's own skip-on-failure semantics for a local `mode="both"`
launch.


In [ ]:
import shlex

cmd_train = [PY, TRAIN_SCRIPT, "--config", CONFIG_PATH] + (shlex.split(EXTRA_ARGS) if EXTRA_ARGS else [])
print("Running (train):", " ".join(shlex.quote(c) for c in cmd_train))
result = subprocess.run(cmd_train)
assert result.returncode == 0, f"{TRAIN_SCRIPT} exited with code {result.returncode}"

eval_extra_flags = shlex.split(EVAL_EXTRA_FLAGS) if EVAL_EXTRA_FLAGS else []
cmd_eval = [PY, EVAL_SCRIPT, "--config", CONFIG_PATH] + eval_extra_flags + (shlex.split(EXTRA_ARGS) if EXTRA_ARGS else [])
print("Running (eval):", " ".join(shlex.quote(c) for c in cmd_eval))
result = subprocess.run(cmd_eval)
assert result.returncode == 0, f"{EVAL_SCRIPT} exited with code {result.returncode}"


## 6. Package results for download

Kaggle keeps `/kaggle/working/` as this notebook version's Output after the session ends —
`backend/kaggle.py`'s `download()` fetches whatever `.zip` file(s) it finds here, so the exact
filename doesn't matter.

In [ ]:
!cd /kaggle/working/segpriors && zip -qr /kaggle/working/results.zip artifacts/ checkpoints/ logs/ -x "*.pth"
!cd /kaggle/working/segpriors && zip -qr /kaggle/working/checkpoints.zip checkpoints/
!ls -lh /kaggle/working/*.zip
